<a href="https://colab.research.google.com/github/ldfha/RotemAI/blob/main/projects/pro15rl/rl4DQN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# DQN : Q-learning을 딥러닝 신경망으로 확장한 강화학습 알고리즘
# Q-table 대신 신경망(Neural Network)으로 Q값을 근사함

# ┌────────────────────────────────────────────────────────────────┐
# │                     DQN 전체 동작 구조                         │
# ├────────────────────────────────────────────────────────────────┤
# │  환경(Environment)                                             │
# │    → 현재 상태(state) 입력                                     │
# │    → Q-Network로 행동별 Q값 예측                               │
# │    → ε-greedy 정책으로 행동(action) 선택                       │
# │    → 환경에 action 실행                                        │
# │    → reward, next_state, done 수신 (경험)                      │
# │    → Replay Buffer에 경험 저장                                 │
# │    → 버퍼에서 랜덤 샘플링 (mini-batch)                        │
# │    → Target Network로 목표 Q값(target) 계산                    │
# │    → Q-Network 학습 (손실 = 예측Q - 목표Q)                     │
# └────────────────────────────────────────────────────────────────┘

# *** DQN 동작 비유 ***
# 학생(Q-Network)이 문제를 풀며 경험 쌓음
#   → 문제은행(Replay Buffer)에 경험 저장 : (s, a, r, s', done)
#   → 문제은행에서 랜덤으로 문제 꺼냄 : mini-batch sampling
#   → 정답지(Target Network)로 목표값 계산 : r + γ * max Q_target(s')
#   → 학생(Q-Network) 업데이트 : loss = (현재 Q값 - 목표 Q값)²

# ~~~ 이전 CartPole 예제를 DQN으로 작업 ~~~
# ┌──────────────────────┬──────────────────────────────────┐
# │   기존 Q-table 방식   │          DQN 방식                │
# ├──────────────────────┼──────────────────────────────────┤
# │ 상태를 이산화해야 함   │ 연속된 상태를 그대로 사용         │
# │ Q-table로 Q값 저장    │ 신경망 모델이 Q값 예측            │
# │ argmax(Q[state])     │ argmax(model.predict(state))     │
# │ Q[s][a] = r+γ*...   │ model.fit()으로 가중치 갱신       │
# └──────────────────────┴──────────────────────────────────┘

!pip install gymnasium[classic-control]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 49.8 MB/s eta 0:00:00


In [2]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense
from tensorflow.keras.optimizers import Adam
import tensorflow as tf
import gymnasium as gym
from collections import deque
import matplotlib.pyplot as plt
import random
import numpy as np

# CartPole-v1 환경 생성
# - 상태(state) : 카트 위치, 카트 속도, 막대 각도, 막대 각속도 (4차원 연속값)
# - 행동(action) : 왼쪽(0) 또는 오른쪽(1) 이동 (2가지)
env = gym.make("CartPole-v1")
num_actions = int(env.action_space.n)        # 2 (왼쪽, 오른쪽)
state_dim = env.observation_space.shape[0]   # 4 (상태 벡터 차원)
print(num_actions, state_dim)                # 2 4

# ── Q-Network 모델 구조 ──────────────────────────────────────────
# 입력 : 현재 상태 (4차원 벡터)
# 은닉층 : Dense(64) → relu × 2
# 출력 : 각 행동에 대한 Q값 (2개, 제한 없는 실수 → linear 활성화)
# 손실 함수 : MSE (예측 Q값 - 목표 Q값)²
def create_model():
    model = Sequential([
        Input(shape=(state_dim, )),
        Dense(units=64, activation='relu'),
        Dense(units=64, activation='relu'),
        Dense(units=num_actions, activation='linear')  # Q값은 실수 범위 → linear
    ])
    model.compile(optimizer=Adam(learning_rate=0.0005), loss='mse')
    return model

# Q-Network (Online Network) : 매 step마다 가중치 업데이트되는 메인 신경망
model = create_model()

# Target Network : 학습 안정성을 위해 가중치를 고정하는 복사본 신경망
# → 목표값(target) 계산에만 사용. 일정 주기마다 Q-Network 가중치를 복사해 갱신
# → 이유 : 같은 네트워크로 예측과 목표를 동시에 계산하면 'moving target' 문제 발생
target_model = create_model()
target_model.set_weights(model.get_weights())  # 초기 가중치 동기화

2 4


In [3]:
# 하이퍼 파라미터
# 할인율
gammer = 0.99          # γ : 미래 보상의 현재 가치 반영 비율 (1에 가까울수록 장기 보상 중시)

# 탐험율 (ε-greedy 전략)
epsilon = 1.0          # 초기 탐험율 : 처음엔 100% 랜덤 행동
epsilon_decay = 0.995  # 매 에피소드마다 탐험율 감소 비율
epsilon_min = 0.05     # 최소 탐험율 : 항상 5%는 랜덤 탐험 유지

# Replay Buffer
batch_size = 64        # 한 번 학습 시 버퍼에서 꺼내는 샘플 수
memory = deque(maxlen=5000)  # 최대 5000개 경험 저장. 초과 시 오래된 것부터 제거 (FIFO)
# 저장 형태 : (state, action, reward, next_state, done)

# 학습 설정
episodes = 50          # 총 에피소드 수 (충분한 학습엔 300~1000 권장)
update_target_every = 5  # Target Network 갱신 주기 (5 에피소드마다 Q-Network 가중치 복사)

reward_list = []       # 에피소드별 총 보상 기록용

In [4]:
for ep in range(episodes):
    state, _ = env.reset()  # 에피소드 시작 : 환경 초기화 → 초기 상태 반환
    done = False
    total_reward = 0

    while not done:
        # 신경망 입력은 배치 형태 필요 : [x1,x2,x3,x4] → [[x1,x2,x3,x4]]
        state_input = np.reshape(state, [1, state_dim])

        # ε-greedy 행동 선택
        # epsilon 확률로 랜덤 탐험, 나머지는 Q-Network 예측값 기반 행동
        if np.random.rand() < epsilon:
            action = np.random.choice(num_actions)       # 탐험 : 랜덤 행동
        else:
            q_values = model.predict(state_input, verbose=0)
            action = np.argmax(q_values)                 # 활용 : Q값 최대 행동 선택

        # 환경에서 다음 상태 및 보상 수신
        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        # 에피소드 종료 시 강한 패널티 부여 → 막대가 쓰러지는 행동 억제
        modified_reward = reward if not done else -10

        # Replay Buffer에 경험 저장
        # 랜덤 샘플링으로 시간적 상관관계를 끊어 i.i.d 근사
        memory.append((state, action, modified_reward, next_state, done))
        state = next_state
        total_reward += reward

        # Q-Network 학습
        # 버퍼에 batch_size 이상 쌓이면 mini-batch 학습 시작
        if len(memory) >= batch_size:
            minibatch = random.sample(memory, batch_size)  # 랜덤 샘플링
            states, targets = [], []

            for s, a, r, s_next, d in minibatch:
                s_input      = np.reshape(s,      [1, state_dim])
                s_next_input = np.reshape(s_next, [1, state_dim])

                # 현재 상태에 대한 Q값 예측 (모든 행동에 대한 예측값)
                # → 선택한 행동(a)의 Q값만 벨만 방정식으로 갱신, 나머지는 유지
                target = model.predict(s_input, verbose=0)[0]  # shape: [[q0,q1]] → [q0,q1]

                if d:
                    # 종료 상태 : 미래 보상 없음
                    target[a] = r
                else:
                    # 벨만 방정식 : Q(s,a) = r + γ * max Q_target(s', a')
                    # Target Network로 다음 상태의 Q값 예측 (학습 안정성)
                    t_next = target_model.predict(s_next_input, verbose=0)[0]
                    target[a] = r + gammer * np.max(t_next)

                states.append(s)       # 입력 상태
                targets.append(target) # 갱신된 Q값 (정답 레이블)

            # 배치 전체를 한 번에 학습 (epochs=1)
            model.fit(np.array(states), np.array(targets), epochs=1, verbose=0)

    reward_list.append(total_reward)

    # ε 감소
    # 학습이 진행될수록 탐험 줄이고 활용 늘림
    if epsilon > epsilon_min:
        epsilon *= epsilon_decay
        epsilon = max(epsilon, epsilon_min)

    # Target Network 가중치 갱신
    # 일정 주기마다 Q-Network → Target Network로 가중치 복사
    if ep % update_target_every == 0:
        target_model.set_weights(model.get_weights())

    if ep % 10 == 0:
        print(f'Episode {ep}: Reward = {total_reward:.1f}, Epsilon = {epsilon:.3f}')

Episode 0: Reward = 21.0, Epsilon = 0.995
Episode 10: Reward = 19.0, Epsilon = 0.946
Episode 20: Reward = 18.0, Epsilon = 0.900
Episode 30: Reward = 35.0, Epsilon = 0.856
Episode 40: Reward = 86.0, Epsilon = 0.814
